In [30]:
import requests
import json
import pandas as pd
import time

In [31]:
#Etape 1
with open ("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)
df_1=pd.json_normalize(data, record_path=["restaurants"], meta = ["ville", "pays"])
df = df_1[["nom", "adresse", "type", "ville", "pays"]]
df_1.head()

,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


In [32]:
#Etape 2
df["note_moyenne"] = df_1["notes"].apply(lambda x:round(sum(x)/len(x),2) if x else None)
df.head()

,nom,adresse,type,ville,pays,note_moyenne
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,4.70
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,4.45
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,4.15
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,4.00


In [33]:
#Etape 3
def coordonnee(adresse) : 
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (https://github.com/kelly-lachance)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try : 
        r = requests.get(url, params = params, headers = headers, timeout=10)
        r.raise_for_status()
        resultat = r.json()
        if not resultat:
            return None, None
        return [float(resultat[0]["lat"]), float(resultat[0]["lon"])]
    except requests.exceptions.RequestException : 
        return None, None
    finally : 
        time.sleep(1)

df["coordonnees"] = df["adresse"].apply(coordonnee)
df["lat"] = df["coordonnees"].str[0]
df["lon"] = df["coordonnees"].str[1]

df.head()





,nom,adresse,type,ville,pays,note_moyenne,coordonnees,lat,lon
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,4.70,"[37.502093, 15.093195]",37.502093,15.093195
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,4.45,"[37.506171, 15.094322]",37.506171,15.094322
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,4.15,"[37.489569, 15.077596]",37.489569,15.077596
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,4.00,"[37.502933, 15.093041]",37.502933,15.093041


In [34]:
#Etape 4
import folium
#coordonnées valides
valide = df[df["coordonnees"].apply(lambda x : x is not None)]
#carte centrée sur le premier restaurant
m = folium.Map([df.iloc[0]["lat"], df.iloc[0]["lon"]], zoom_start=14)

for _,ligne in valide.iterrows() : 
    folium.Marker(location=[ligne["lat"],ligne["lon"]], popup=f"{ligne["nom"]}, Note: {ligne["note_moyenne"]}").add_to(m)
m

In [35]:
#Etape 5
df.drop(columns=["coordonnees"], inplace=True)
df.to_csv('../data/restaurants_geocodes.csv', index=False)

C:\Users\kelly\AppData\Local\Temp\ipykernel_10120\900669358.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["coordonnees"], inplace=True)
